In [ ]:
import numpy as np
from pathlib import Path

REAL_COMPOSITE_PATH = Path(
    "/home/maria/SelfStudyThesis/data/"
    "allen_natural_scenes_four_class_composite.npy"
)

SYNTHETIC_PATH = Path(
    "/home/maria/SelfStudyThesis/results/"
    "four_class_logistic_design_loo_image_probs/"
    "synthetic_neural_activity_image_probs_loo.npy"
)

BINARY_TRIAL_COMPOSITE_PATH = Path(
    "/home/maria/SelfStudyThesis/data/"
    "allen_natural_scenes_four_class_binary_trials_composite.npy"
)

def load_real_and_synthetic():
    print()
    print("#" * 80)
    print("Loading real and supervised synthetic matrices")
    print("#" * 80)
    print(f"Real composite: {REAL_COMPOSITE_PATH}")
    print(f"Synthetic:      {SYNTHETIC_PATH}")

    data = np.load(REAL_COMPOSITE_PATH, allow_pickle=True).item()

    X_real = np.asarray(data["X"], dtype=np.float64)
    labels = np.asarray(
        data["stimulus_metadata"]["label"], dtype=np.int64
    ).ravel()

    if X_real.shape[1] == len(labels):
        X_real = X_real.T
    elif X_real.shape[0] == len(labels):
        pass
    else:
        raise ValueError(
            f"Cannot align real X={X_real.shape} with labels={labels.shape}."
        )

    X_synth = np.asarray(
        np.load(SYNTHETIC_PATH, allow_pickle=True),
        dtype=np.float64,
    )

    if X_synth.shape == X_real.T.shape:
        X_synth = X_synth.T
    elif X_synth.shape == X_real.shape:
        pass
    else:
        raise ValueError(
            f"Real and synthetic shapes do not align: "
            f"real={X_real.shape}, synthetic={X_synth.shape}."
        )

    keep_images = labels >= 0
    X_real = X_real[keep_images]
    X_synth = X_synth[keep_images]
    labels_kept = labels[keep_images]
    original_image_indices = np.flatnonzero(keep_images)

    finite = np.all(np.isfinite(X_real), axis=0)
    finite &= np.all(np.isfinite(X_synth), axis=0)

    real_var = np.var(X_real, axis=0, ddof=0)
    synth_var = np.var(X_synth, axis=0, ddof=0)

    # Keep neurons that can contribute to either geometry, while ensuring
    # StandardScaler and PCA remain numerically valid for the real matrix.
    neuron_mask = finite & (real_var > EPS) & (synth_var > EPS)

    X_real = X_real[:, neuron_mask]
    X_synth = X_synth[:, neuron_mask]

    print()
    print("Aligned data:")
    print(f"  real:               {X_real.shape} images × neurons")
    print(f"  synthetic:          {X_synth.shape} images × neurons")
    print(f"  retained images:    {len(labels_kept)}")
    print(f"  retained neurons:   {int(neuron_mask.sum())}")
    print(f"  removed neurons:    {int((~neuron_mask).sum())}")

    print()
    print("Label counts:")
    values, counts = np.unique(labels_kept, return_counts=True)
    for value, count in zip(values, counts):
        print(
            f"  {int(value):>2} = "
            f"{LABEL_NAMES.get(int(value), 'UNKNOWN'):<16} "
            f"n={int(count)}"
        )

    return (
        X_real,
        X_synth,
        labels_kept,
        original_image_indices,
        neuron_mask,
    )